In [ ]:
# Sam Brown
# sam_brown@mines.edu
# Poster figures

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import pandas as pd
import numpy as np
import my_lib.funcs
import scipy

import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature


df = pd.read_csv("/Users/sambrown04/Documents/SURF/Preproc_data/averages_events_2011-13.csv")
df = df[1:]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Scatter 
scatter = ax.scatter(
    df['total_delta'],
    df['mins_since'],
    c=df['tide_height'],   # color by form_factors
    cmap='viridis',
    s=50
)


cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label('Tide Heights (cm)', fontsize = 25)
cbar.ax.tick_params(labelsize=14) 

ax.set_xlabel('Delta Among Gz stations (m)', fontsize = 23)
ax.set_ylabel('Time Since last Event (min)', fontsize = 24)

ax.tick_params(axis='x', labelsize=15)  # X-axis ticks
ax.tick_params(axis='y', labelsize=15)  # Left Y-axis ticks

ax.grid(True)

plt.savefig("lintrend.png", dpi=300, bbox_inches="tight", pad_inches=0.5)

In [ ]:
# Figure that shows tidal movements to help convey form factor.
evts_path = "/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas"

events_list = my_lib.funcs.load_evt(evts_path)

tide_time = my_lib.funcs.get_tide_height(365, -165535.17251369057, -597424.3876911158 , "2013-01-01 00:00:00")

In [ ]:
plot_tide_period(tide_time, "2013-03-01 00:00:00")

In [ ]:
def plot_tide_period(df, start_time):

    start_time = pd.to_datetime(start_time)
    end_time = start_time + pd.Timedelta(days=10)
    
    mask = (df['time'] >= start_time) & (df['time'] < end_time)
    subset = df.loc[mask]
    
    # Plot
    plt.figure(figsize=(12,6))
    plt.plot(subset['time'], subset['tide_height'], label="Tide Height", lw = 4)
    plt.xlabel("Time", size = 30)
    plt.ylabel("Tide Height (cm)", fontsize = 30)
    plt.title("Tide Height", size = 30)
    plt.grid(True)
    plt.tight_layout()
    plt.xticks(fontsize=20, rotation=30)
    plt.yticks(fontsize=20)
    
    plt.savefig("tide.png", dpi=300, bbox_inches="tight", pad_inches=0.5)

In [ ]:
t = np.linspace(0, 24, 1000)

# Standardized tidal signals (unitless height, between ~ -1 and 1.5)
semidiurnal = np.sin(2 * np.pi * t / 12)             # 2 highs & 2 lows
diurnal = 0.8 * np.sin(2 * np.pi * t / 24)           # 1 high & 1 low
mixed = np.sin(2 * np.pi * t / 12) + 0.5 * np.sin(2 * np.pi * t / 24)  # unequal highs

# Create subplots: 1 row, 3 columns
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

# Semidiurnal
axes[0].plot(t, semidiurnal, color="navy", linewidth=4)
axes[0].set_title("Semidiurnal", color="navy", fontsize = 30)
axes[0].set_xlabel("Time (h)", fontsize = 30)
axes[0].set_ylabel("Tide Height", fontsize = 30)
axes[0].axhline(0, color="black", linestyle="--", linewidth=4)
axes[0].set_xlim(0, 24)
axes[0].set_ylim(-1.5, 1.5)
axes[0].tick_params(axis='x', labelsize=20)
axes[0].tick_params(axis='y', labelsize=20)
axes[0].grid(True, linestyle="--", alpha=0.6)

# Diurnal
axes[1].plot(t, diurnal, color="red", linewidth=4)
axes[1].set_title("Diurnal", color="red", fontsize = 30)
axes[1].set_xlabel("Time (h)", fontsize = 30)
axes[1].axhline(0, color="black", linestyle="--", linewidth=4)
axes[1].set_xlim(0, 24)
axes[1].set_ylim(-1.5, 1.5)
axes[1].tick_params(axis='x', labelsize=20)
axes[1].tick_params(axis='y', labelsize=20)
axes[1].grid(True, linestyle="--", alpha=0.6)

# Mixed semidiurnal
axes[2].plot(t, mixed, color="green", linewidth=4)
axes[2].set_title("Mixed semidiurnal", color="green", fontsize = 30)
axes[2].set_xlabel("Time (h)", fontsize = 30)
axes[2].axhline(0, color="black", linestyle="--", linewidth=4)
axes[2].set_xlim(0, 24)
axes[2].set_ylim(-1.5, 1.5)
axes[2].tick_params(axis='x', labelsize=20)
axes[2].tick_params(axis='y', labelsize=20)

axes[2].grid(True, linestyle="--", alpha=0.6)

# Tidy layout
plt.tight_layout()

plt.savefig("diurn.png", dpi=300, bbox_inches="tight", pad_inches=0.5)

In [ ]:
df = pd.read_csv('station_standards.csv')


proj = ccrs.SouthPolarStereo()

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=proj)

# Split data
stations_of_interest = ["gz12", "la08", "la05"]
df_focus = df[df['Station'].isin(stations_of_interest)]
df_background = df[~df['Station'].isin(stations_of_interest)]

# Background, other stations gray
ax.scatter(df_background['x_cor'], df_background['y_cor'],
           color='gray', s=30, alpha=0.5, transform=proj, zorder=1)

# Foreground
scatter = ax.scatter(df_focus['x_cor'], df_focus['y_cor'],
                     c=df_focus['slip_severity'], cmap='viridis',
                     s=120, edgecolor='black', linewidth=0.8,
                     transform=proj, zorder=2)

# Labels
for i, row in df_focus.iterrows():
    ax.text(row['x_cor'], row['y_cor'], row['Station'],
            transform=proj, fontsize=20, fontweight='bold',
            color='darkred', ha='center', va='bottom', zorder=3)

# maps feats
land = cfeature.NaturalEarthFeature('physical', 'land', '10m',
                                    edgecolor='none', facecolor='lightgray')
ice = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_polys', '10m',
                                   edgecolor='none', facecolor='lightblue')

ax.add_feature(land, zorder=0)
ax.add_feature(ice, zorder=0)
ax.coastlines(resolution='10m', linewidth=0.6, zorder=1)

# Gridlines
ax.gridlines(linewidth=0.5, color='gray', alpha=0.5, linestyle='--')

# Zoom extent around focus stations
x_min, x_max = df_focus['x_cor'].min(), df_focus['x_cor'].max()
y_min, y_max = df_focus['y_cor'].min(), df_focus['y_cor'].max()
pad = 200000
ax.set_extent([x_min - pad, x_max + pad, y_min - pad, y_max + pad], crs=proj)

# Axis labels in meters
ax.set_xlabel("X coordinate (meters, South Polar Stereographic)")
ax.set_ylabel("Y coordinate (meters, South Polar Stereographic)")

# Colorbar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
cbar.set_label('Slip severity', fontsize = 25)
cbar.ax.tick_params(labelsize=20)

# Inset Antarctica
inset_ax = fig.add_axes([0.5, 0.55, 0.25, 0.25], projection=proj)
inset_ax.set_extent([-3e6, 3e6, -3e6, 3e6], crs=proj)

inset_ax.add_feature(land, zorder=0)
inset_ax.add_feature(ice, zorder=0)
inset_ax.coastlines(resolution='10m', linewidth=0.6, zorder=1)

# Rectangle showing zoom area
inset_ax.plot([x_min - pad, x_max + pad, x_max + pad, x_min - pad, x_min - pad],
              [y_min - pad, y_min - pad, y_max + pad, y_max + pad, y_min - pad],
              transform=proj, color='red', linewidth=1.5)

plt.savefig("Map.png", dpi=300)
